# Demo 2: Visualization of atomic environments with cl-MDS
---


In [ ]:
import numpy as np
import cluster_mds as clmds
import matplotlib.pyplot as plt
from weas_widget import WeasWidget

## Quick review of Demo 1
---

```python
import module_name

# initialize the method
method = module_name.class_name( input_parameters )

# run the method
method.get_embedding_name( main_parameters )

# get ouptut
method.output
```

In [ ]:
# write here the simplest workflow with cl-MDS (live demo)


What is happening under the hood?

In [ ]:
# show clMDS class attributes

# from initialization
method.atoms
method.species_list

method.all_env
method.n_env
method.sparse_list

# from calculations to prepare the input data
method.descriptor  #method.build_descriptor
method.dist_matrix #method.build_dist_matrix
method.zeta

# from running clMDS
method.sparse_coordinates
method.sparse_medoids
method.sparse_cluster_indices





## Analysing a database of Ge-Sb-Te materials
---


### 1. Understanding the main parameters

In [ ]:
# input file
atoms_file = "GST_training_data.xyz"

# directories to store the results
dirname = './results_demo_2/'
dir_medoids = dirname + 'medoids/'

# descriptor information
desc_type = "quippy_soap_turbo"
Z = ["Ge", "Sb", "Te"]
nmax = 8; lmax = 8
rcut_hard = 4.; rcut_soft = rcut_hard - 0.5
at_sr = [0.5, 0.5, 0.5]; at_srs = [0., 0., 0.]
at_st = [0.5, 0.5, 0.5]; at_sts = [0., 0., 0.]
desc_string = {z:'soap_turbo alpha_max={%i %i %i %i %i} l_max=%i rcut_soft=%.2f \
                 rcut_hard=%.2f atom_sigma_r={%.2f %.2f %.2f} atom_sigma_t={%.2f %.2f %.2f} \
                 atom_sigma_r_scaling={%.2f %.2f %.2f} atom_sigma_t_scaling={%.2f %.2f %.2f} \
                 radial_enhancement=1 compress_mode=trivial \
                 amplitude_scaling={1. 1. 1.} basis="poly3gauss" \
                 scaling_mode="polynomial" species_Z={32 51 52} n_species=3 \
                 central_index=%i central_weight={1. 1. 1.}'
                 % (nmax, nmax, nmax, lmax, rcut_soft, rcut_hard, at_sr[0],
                    at_sr[1], at_sr[2], at_st[0], at_st[1], at_st[2], at_srs[0], at_srs[1],
                    at_srs[2], at_sts[0], at_sts[1], at_sts[2], i+1)
                 for i, z in enumerate(Z)}

# embedding info
do_species = ['Ge','Sb','Te']

# clustering
hierarchy = [15,1]

### 2. Sparsification

In [ ]:
# Generate a custom list/array of sparse indices
filename_sparse = None

if filename_sparse is not None:
    custom_sparse = np.loadtxt(filename_sparse, dtype=int)
else:
#   Compute a custom sparse set using sparsify_module
    import sparsify_module as spmod
    
    n_sparse = 1000 # reference size of the sparse set
    P_med = 15 # approx. percentage of medoids in the sparse set
               # lower it (15-20) for databases with more than 30000 atoms
#   (1) only medoids in the sparse set
#    custom_sparse = spmod.sparsify_kmedoids( atoms=atoms_file, descriptor=descriptor, do_species=do_species,
#                 descriptor_string=desc_string, max_n_sparse=n_sparse, percentage_med=P_med )

#   (2) medoids + random points in the sparse set
#    custom_sparse = spmod.sparsify_rand_and_kmedoids( n_sparse, atoms=atoms_file, descriptor=descriptor,
#                  descriptor_string=desc_string, do_species=do_species, percentage_med=P_med )

#   (3) optimized version of (1), (2) for a given number of clusters 
#       it ensures a minimum number of points per cluster in the sparse set, improving cl-MDS performance
    custom_sparse = spmod.sparsify_cluster_size( n_sparse, hierarchy[0], atoms=atoms_file, 
                                                descriptor=descriptor, descriptor_string=desc_string, 
                                                do_species=do_species, percentage_med=P_med, 
                                                min_cluster_size=5, max_iter=15 )

#   It is recommended to save the sparse set for further uses/testing parameters
    np.savetxt('improved_sparse_set.txt', custom_sparse, fmt='%i')

### 3. cl-MDS calculations

In [ ]:
# Initialize clMDS class passing our own descriptor string
method = clmds.clMDS(atoms=atoms_file, descriptor=descriptor,
                     descriptor_string=desc_string, do_species=do_species,
                     sparsify=custom_sparse)
# Compute 2-dim. representation of the sparse set
method.cluster_MDS(hierarchy,
                 # kmedoids calculations (n. iterations, initialization)
                 iter_med=1000, 
                 init_medoids="isolated", n_iso_med=2,
                 # embedding of clusters (n. iterations, n. CPUS, custom weights)
                 n_init_mds_cluster=100, 
                 n_jobs_cluster=8, 
                 weight_cluster_mds=8,
                 # anchor points selection
                 param_anchor=[80,90,90], 
                 # anchor points embedding (n. iterations, n. CPUS, custom weights)
                 n_init_mds_anchor=3500, 
                 n_jobs_anchor=8, 
                 weight_anchor_mds=2, eta=0)

Y = method.sparse_coordinates
C = method.sparse_cluster_indices
M = method.sparse_medoids

### 4. Estimation of coordinates for additional points

In [ ]:
# Estimate the coordinates for the rest of the database
Y_estim = method.get_estim_coordinates(n_steps=10)
C_estim = Y_estim[:,2].astype(int)

### 5. Visualization

In [ ]:
# Save to file
method.save_to_file(dir=dirname)

In [ ]:
# show plot
fig, ax = plt.subplots()
ax.scatter(Y_estim[:, 0], Y_estim[:, 1], c=C_estim, cmap='nipy_spectral',
            alpha=0.3)
ax.scatter(Y[:, 0], Y[:, 1], c=C, cmap='nipy_spectral')
ax.scatter(Y[M, 0], Y[M, 1], color='black', label='medoids')
ax.set_xlabel(r'cl-MDS coordinate 1')
ax.set_ylabel(r'cl-MDS coordinate 2')

plt.legend()
#plt.savefig(dirname + 'clmds_plot_advanced.png', format='png', dpi=300)
plt.show()



In [ ]:
# Visualize with ovito the atomic environments corresponding to the medoids
if method.atoms is not None:
    method.medoids_to_xyz(dir=dir_medoids, carve_radius=3.5, render=True)
    
    from ase.io import read
    
    for i in range(0, len(M)):
        prit("Medoid n. %i" % i)
        atoms = read("medoid_%i.xyz" % i)
        viewer = WeasWidget()
        viewer.avr.model_style = 1 # ball & stick mode
        viewer.from_ase(atoms)
        viewer.avr.show_bonded_atoms = True # show bonds across periodic boundaries
        viewer

### 6. Adding simulation data on top of the embedding

In [ ]:
# local energies
E_local

In [1]:
# Hirshfeld volumes


### 7. Iterative training analysis (evolution of data)

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

# retrieve the indexes of the atomic env coming from different iterations
indexes = {}
l=0
for ats in method.atoms:
    label = ats.info['config_type']
    if 'iter' in label:
        indexes[label].setdefault([]) += range(l, l + len(ats),1)
        

@interact(n_iter=(1, 8, 1))
def interactive_vis(n_iter=1):
    label_iter = 'iter_%i' % n_iter
    I = indexes[label_iter]
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(Y_estim[:, 0], Y_estim[:, 1], c=E_local, 
               cmap='nipy_spectral', alpha=0.3)
    ax.scatter(Y_estim[I, 0], Y_estim[I, 1], color='black', 
               label=label_iter)
    ax.set_title(f"n_iter={n_iter:i}")
    plt.show()
